In [6]:
import sys

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent

sys.path.append(str(ROOT_DIR))
print(ROOT_DIR)

/home/kenzi/Desktop/S3/ML_FOR_CS/SENTINEL-U


In [3]:
from src.data.dataset import load_dataset
from src.data.preprocessing import check_duplicates,check_invalid_logins,check_missing_values,check_negative_values,convert_timestamp

from src.models.statistical.global_baseline import GlobalBaseline

from src.evaluation.metrics import (
    evaluate,
    precision_at_k
)
def data_clean(df):
    check_negative_values(df)
    check_missing_values(df)
    check_invalid_logins(df)
    check_duplicates(df)
    return df

In [10]:
ROOT = ROOT_DIR/"Data"/"raw"/"auth_hourly.csv"
df_raw = load_dataset(ROOT)

df = data_clean(df_raw)

df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

print(df.shape)

display(df.head())

(264888, 19)


,timestamp,user_id,peer_group,hour,day_of_week,is_weekend,off_hours,logins,failed_logins,distinct_ips,distinct_hosts,mb_out,dns_queries,country,foreign_country,split,benign_event,is_attack,attack_type
0,2026-09-07,u000,student,0,0,0,1,1,0,1,1,0.66,10,MA,0,train,none,0,none
1,2026-09-07,u001,student,0,0,0,1,0,0,0,0,0.84,53,MA,0,train,none,0,none
2,2026-09-07,u002,student,0,0,0,1,0,0,0,0,0.40,1,MA,0,train,none,0,none
3,2026-09-07,u003,student,0,0,0,1,0,0,0,0,1.02,38,MA,0,train,none,0,none
4,2026-09-07,u004,student,0,0,0,1,0,0,0,0,0.17,1,MA,0,train,none,0,none


In [11]:
EVALUATION_COLUMNS = [
    "is_attack",
    "attack_type",
    "benign_event"
]

In [12]:
train_df = df[
    df["split"] == "train"
].copy()

validation_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("Train :", train_df.shape)

print("Validation :", validation_df.shape)

print("Test :", test_df.shape)

Train : (130488, 19)
Validation : (33600, 19)
Test : (100800, 19)


In [13]:
detector = GlobalBaseline(n_sigma=3)

detector.fit(train_df)

In [14]:
display(detector.statistics_)

,mean,std
failed_logins,0.009988,0.083574
mb_out,0.733968,0.604658
dns_queries,2.595428,1.062313


In [15]:
print("Thresholds:")

display(detector.thresholds_)

Thresholds:


failed_logins    0.260709
mb_out           2.547942
dns_queries      5.782367
dtype: float64

In [16]:
test_scores = detector.score_samples(test_df)

test_alerts = detector.predict(test_df)

In [17]:
test_results = test_df.copy()

test_results["anomaly_score"] = test_scores

test_results["alert"] = test_alerts

display(
    test_results[
        [
            "user_id",
            "timestamp",
            "failed_logins",
            "mb_out",
            "dns_queries",
            "anomaly_score",
            "alert"
        ]
    ].head(20)
)

,user_id,timestamp,failed_logins,mb_out,dns_queries,anomaly_score,alert
164088,u000,2026-10-12,0,0.77,20,0.422752,False
164089,u001,2026-10-12,0,2.71,33,0.954363,False
164090,u002,2026-10-12,0,0.38,3,-0.119514,False
164091,u003,2026-10-12,0,0.94,18,0.328539,False
164092,u004,2026-10-12,0,0.26,4,-0.119514,False
164093,u005,2026-10-12,0,0.63,16,0.223838,False
164094,u006,2026-10-12,0,0.37,7,-0.119514,False
164095,u007,2026-10-12,0,0.40,15,0.166769,False
164096,u008,2026-10-12,0,0.80,15,0.166769,False
164097,u009,2026-10-12,0,0.33,18,0.328539,False


In [18]:
results = evaluate(
    alerts=test_alerts,
    test_df=test_df
)

for metric, value in results.items():

    print(f"\n{metric}:")

    print(value)


FPR:
0.01931877546755825

Benign Alert Rate:
0.18855693348365277

Total Alerts:
2832

Alerts Per Day:
134.85714285714286

Recall Per Type:
{'mimicry': np.float64(0.04961240310077519), 'low_slow_exfil': np.float64(0.32653061224489793), 'account_takeover': np.float64(1.0), 'dns_tunnel': np.float64(0.0), 'brute_force': np.float64(1.0), 'night_exfil': np.float64(1.0)}
